# Mixed-precision reasoning MVP analysis
Load `../runs/summary.json`, `comparison.jsonl`, and the feature Parquet files here after a run.

In [ ]:
from pathlib import Path
import json, pandas as pd
runs = Path('../runs')
summary = json.loads((runs / 'summary.json').read_text())
summary

In [ ]:
features = pd.read_parquet(runs / 'example_features.parquet')
eligible = features.query('eligible_fp_correct == 1').copy()
eligible['entropy_bin'] = pd.qcut(eligible.max_entropy, q=min(10, len(eligible)), duplicates='drop')
curve = eligible.groupby('entropy_bin', observed=True).agg(mean_score=('max_entropy', 'mean'), failure_probability=('target_quantization_failure', 'mean'), count=('example_id', 'size'))
ax = curve.plot(x='mean_score', y='failure_probability', marker='o', legend=False, title='Sensitivity score vs quantization-failure probability')
ax.set(xlabel='Max quantized-token entropy', ylabel='P(FP correct, quantized wrong)')
ax.figure.savefig(runs / 'sensitivity_vs_failure.png', bbox_inches='tight', dpi=160)
curve

In [ ]:
predictor_metrics = json.loads((runs / 'predictor_metrics.json').read_text())
pd.DataFrame(predictor_metrics.get('results', {})).T